# Filter Design Foundations
SciPy designs a filter in one call (`firwin`, `butter`, ...). This notebook opens up the recipes behind those calls:
1. **FIR filters by the window method**: start from the ideal low-pass impulse response (a sinc), cut it to a finite length, and taper it with a window.
2. **Spectral inversion**: turn a linear-phase low-pass FIR into a high-pass.
3. **IIR filters from analog prototypes**: the z-transform, the bilinear transform, and frequency pre-warping.

`digital_filters.ipynb` then compares complete FIR and IIR designs made with SciPy.

Background reading: Robert Bristow-Johnson, *Audio EQ Cookbook* (the biquad formulas in `audiodsp.rbj_biquad`).

In [ ]:
import warnings
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
from IPython import display
import ipywidgets as widgets
from ipywidgets import interactive
from audiodsp import rbj_biquad

%matplotlib inline
plt.style.use('seaborn-v0_8')
plt.rcParams['figure.figsize'] = (14, 5)

## 1. FIR filters by the window method
An ideal low-pass filter passes everything below the cutoff $f_c$ and nothing above it. Its impulse response is a sinc that extends forever in both directions:

$$h_{ideal}[n] = 2 f_c\, \mathrm{sinc}(2 f_c n), \qquad -\infty < n < \infty$$

Here $f_c$ is in cycles per sample, so $f_c = 0.25$ means $f_s/4$. A real filter must be finite and causal, so the sinc is cut to $N = M + 1$ taps centered on $n = M/2$.

Cutting the sinc is the same as multiplying it by a rectangular window. Multiplying in time convolves in frequency, so the ideal brick-wall response is smeared by the window's spectrum. Ripples appear on both sides of the cutoff (the **Gibbs phenomenon**), and they do not shrink as $N$ grows. They only get narrower. Tapering the ends of the sinc with a smooth window lowers the ripples, at the cost of a wider transition band.

Two details matter:
- The window must be **symmetric** (`signal.get_window(..., fftbins=False)`, or `sym=True` in `scipy.signal.windows`). Symmetric taps give exactly linear phase. `fftbins=True` / `sym=False` returns the periodic version of the window, which is meant for spectral analysis, and its taps are not symmetric.
- The taps are scaled to sum to 1, so the gain at 0 Hz is exactly 1 (0 dB).

In [ ]:
N, FC = 61, 0.25
M = N - 1
n = np.arange(N) - M / 2
h_ideal = 2 * FC * np.sinc(2 * FC * n)

def windowed_sinc(window, n_taps=N, fc=FC):
    k = np.arange(n_taps) - (n_taps - 1) / 2
    h = 2 * fc * np.sinc(2 * fc * k) * signal.get_window(window, n_taps, fftbins=False)
    return h / h.sum()

def response_db(h, n_points=8192):
    w, H = signal.freqz(h, worN=n_points)
    return w / (2 * np.pi), 20 * np.log10(np.abs(H) + 1e-12)   # frequency in cycles per sample

def first_null_and_peak_after(f, mag, start):
    """Index of the first local minimum after frequency `start`, and the largest level beyond it."""
    i = np.searchsorted(f, start)
    while i + 1 < len(mag) and mag[i + 1] < mag[i]:
        i += 1
    return f[i], mag[i:].max()

WINDOWS = ['boxcar', 'hann', 'hamming', 'blackman']
print(f'{"Window":<10}{"Window side lobe":>18}{"Filter first null":>19}{"Filter stopband peak":>22}')
for name in WINDOWS:
    # The window's own spectrum: largest side lobe after its main lobe
    w = signal.get_window(name, N, fftbins=False)
    W = 20 * np.log10(np.abs(np.fft.rfft(w, 1 << 16)) / w.sum() + 1e-12)
    _, side_lobe = first_null_and_peak_after(np.fft.rfftfreq(1 << 16), W, 0)
    # The filter built with it
    f, mag = response_db(windowed_sinc(name))
    null_f, stop_peak = first_null_and_peak_after(f, mag, FC)
    print(f'{name:<10}{side_lobe:>15.1f} dB{null_f:>19.3f}{stop_peak:>19.1f} dB')

The window's side lobe level and the filter's stopband level are different numbers. The filter's response at each frequency is the window spectrum summed over a band as wide as the passband, and the side lobes partly cancel in that sum. So the filter's stopband is always lower than the window's own side lobe. The first null shows the cost: windows with lower side lobes have a wider main lobe, so the filter's transition band is wider.

Change the window and the number of taps below. More taps narrow the transition band for any window, but only the window choice lowers the stopband.

In [ ]:
def plot_windowed_sinc(window='hann', n_taps=61):
    h = windowed_sinc(window, n_taps)
    h_rect = windowed_sinc('boxcar', n_taps)
    f, mag = response_db(h)
    _, mag_rect = response_db(h_rect)

    fig, axes = plt.subplots(1, 2, figsize=(16, 4.5))
    axes[0].plot(h_rect, 'o-', color='0.7', markersize=3, label='rectangular (truncated sinc)')
    axes[0].plot(h, 'o-', markersize=3, label=str(window))
    axes[0].set_title(f'{n_taps} taps')
    axes[0].set_xlabel('Tap index')
    axes[0].legend()
    axes[1].plot(f, mag_rect, color='0.7', label='rectangular')
    axes[1].plot(f, mag, label=str(window))
    axes[1].axvline(FC, color='k', linestyle=':', label='cutoff')
    axes[1].set_ylim(-120, 5)
    axes[1].set_xlabel('Frequency (cycles per sample; 0.5 = Nyquist)')
    axes[1].set_ylabel('dB')
    axes[1].set_title('Magnitude response')
    axes[1].legend(loc='lower left')
    plt.tight_layout()
    plt.show()

display.display(interactive(
    plot_windowed_sinc,
    window=widgets.Dropdown(options=[(w, w) for w in WINDOWS] + [('kaiser, beta = 8', ('kaiser', 8.0))],
                            value='hann', description='Window:'),
    n_taps=widgets.IntSlider(value=61, min=11, max=301, step=10, description='Taps:'),
))

The Kaiser window has a parameter $\beta$ that trades side lobe level against main lobe width, which is why `firwin` and `kaiserord` use it.

## 2. Spectral inversion: low-pass to high-pass
A linear-phase FIR with odd length $N$ delays every frequency by exactly $M/2$ samples. Its response is $H_{LP}(e^{j\omega}) = e^{-j\omega M/2} A(\omega)$, where $A(\omega)$ is real. A unit impulse at the center tap, $\delta[n - M/2]$, is an all-pass with the same delay. Subtracting gives

$$h_{HP}[n] = \delta[n - M/2] - h_{LP}[n] \quad\Longrightarrow\quad H_{HP}(e^{j\omega}) = e^{-j\omega M/2}\,\big(1 - A(\omega)\big)$$

Where the low-pass passes ($A = 1$) the high-pass blocks, and the other way round. This needs an odd number of taps, so that the center tap exists, and a low-pass with gain exactly 1 at 0 Hz.

Because both terms are symmetric, the high-pass also has exactly linear phase: a constant group delay of $M/2 = 30$ samples. The same design with the periodic window (`fftbins=True`) is close, but its group delay wanders by a few thousandths of a sample. Small here, but the exact version costs nothing.

In [ ]:
h_lp = windowed_sinc('hann')
center = np.zeros(N)
center[M // 2] = 1
h_hp = center - h_lp

f, mag_lp = response_db(h_lp)
_, mag_hp = response_db(h_hp)
# The same high-pass built from the periodic (asymmetric) Hann window
h_lp_periodic = h_ideal * signal.get_window('hann', N, fftbins=True)
h_lp_periodic /= h_lp_periodic.sum()

with warnings.catch_warnings():
    warnings.simplefilter('ignore')   # group delay is undefined at the high-pass's zero at 0 Hz
    w_gd, gd = signal.group_delay((h_hp, [1.0]), w=4096)
    _, gd_periodic = signal.group_delay((center - h_lp_periodic, [1.0]), w=4096)
f_gd = w_gd / (2 * np.pi)
passband = f_gd > FC + 0.05

fig, axes = plt.subplots(1, 2, figsize=(16, 4.5))
axes[0].plot(f, mag_lp, label='low-pass (Hann)')
axes[0].plot(f, mag_hp, label='high-pass = impulse - low-pass')
axes[0].axvline(FC, color='k', linestyle=':')
axes[0].set_ylim(-100, 5)
axes[0].set_xlabel('Frequency (cycles per sample)')
axes[0].set_ylabel('dB')
axes[0].legend()
axes[1].plot(f_gd[passband], gd[passband], label='symmetric window')
axes[1].plot(f_gd[passband], gd_periodic[passband], label='periodic window')
axes[1].set_xlabel('Frequency (cycles per sample)')
axes[1].set_ylabel('Samples')
axes[1].set_title('High-pass group delay in its passband')
axes[1].legend()
plt.tight_layout()
plt.show()
print(f'Group delay, symmetric window: {gd[passband].min():.3f} to {gd[passband].max():.3f} samples (M/2 = {M // 2})')
print(f'Group delay, periodic window:  {gd_periodic[passband].min():.3f} to {gd_periodic[passband].max():.3f} samples')

## 3. IIR filters from analog prototypes
### The z-transform
The z-transform of an impulse response is $H(z) = \sum_n h[n]\, z^{-n}$. Each factor $z^{-1}$ is a one-sample delay, so the difference equation

$$y[n] = b_0 x[n] + b_1 x[n-1] + b_2 x[n-2] - a_1 y[n-1] - a_2 y[n-2]$$

has the transfer function

$$H(z) = \frac{b_0 + b_1 z^{-1} + b_2 z^{-2}}{1 + a_1 z^{-1} + a_2 z^{-2}}$$

This second-order section is the **biquad**, the building block of IIR filters and equalizers. Evaluating $H(z)$ on the unit circle, $z = e^{j\omega}$ with $\omega = 2\pi f / f_s$, gives the frequency response. `digital_filters.ipynb` section 4 plots its poles and zeros.

### The bilinear transform
Decades of analog filter design (Butterworth, Chebyshev, elliptic, equalizer circuits) is written as transfer functions $H(s)$ of the Laplace variable $s$. The **bilinear transform** turns one into a digital $H(z)$ by substituting

$$s = 2 f_s\, \frac{1 - z^{-1}}{1 + z^{-1}}$$

It keeps stable filters stable and maps the whole analog frequency axis (0 to infinity) onto the digital one (0 to $f_s/2$). That squeeze is not even. An analog frequency $f_a$ lands at the digital frequency $f_d$ where

$$f_a = \frac{f_s}{\pi} \tan\!\left(\frac{\pi f_d}{f_s}\right)$$

Low frequencies map almost one to one. Near Nyquist, an infinite range of analog frequencies is compressed into the last part of the digital band. This is **frequency warping**.

**Pre-warping** puts one chosen frequency exactly where it belongs: design the analog prototype at $f_a = \frac{f_s}{\pi}\tan(\pi f_0 / f_s)$ instead of at $f_0$, and after the transform it lands on $f_0$. SciPy's `butter`, `cheby1`, `iirfilter` and the like pre-warp their cutoff frequencies automatically when given `fs`. The Audio EQ Cookbook formulas are pre-warped bilinear transforms too.

Below, an analog peaking equalizer (+6 dB at 10 kHz, Q = 2) is converted three ways.

In [ ]:
fs, f0, Q, gain_db = 44100, 10000, 2.0, 6.0
A = 10 ** (gain_db / 40)

def analog_peaking(f_center):
    """H(s) = (s^2 + s (A/Q) W + W^2) / (s^2 + s W / (A Q) + W^2), W = 2 pi f_center."""
    W = 2 * np.pi * f_center
    return [1, A / Q * W, W**2], [1, W / (A * Q), W**2]

b_s, a_s = analog_peaking(f0)
b_plain, a_plain = signal.bilinear(b_s, a_s, fs)
f_prewarped = fs / np.pi * np.tan(np.pi * f0 / fs)
b_pw, a_pw = signal.bilinear(*analog_peaking(f_prewarped), fs)
b_rbj, a_rbj = rbj_biquad('peaking', f0, fs, q=Q, gain_db=gain_db)

freqs = np.linspace(20, fs / 2, 20000)
_, H_analog = signal.freqs(b_s, a_s, worN=2 * np.pi * freqs)
_, H_plain = signal.freqz(b_plain, a_plain, worN=freqs, fs=fs)
_, H_pw = signal.freqz(b_pw, a_pw, worN=freqs, fs=fs)

fig, axes = plt.subplots(1, 2, figsize=(16, 4.5))
f_digital = np.linspace(0, fs / 2 * 0.999, 1000)
axes[0].plot(f_digital / 1000, fs / np.pi * np.tan(np.pi * f_digital / fs) / 1000, label='bilinear transform')
axes[0].plot(f_digital / 1000, f_digital / 1000, 'k:', label='no warping')
axes[0].set_ylim(0, 60)
axes[0].set_xlabel('Digital frequency (kHz)')
axes[0].set_ylabel('Analog frequency it came from (kHz)')
axes[0].set_title('Frequency warping at fs = 44.1 kHz')
axes[0].legend()
for H, label, style in [(H_analog, 'analog prototype', 'k-'),
                        (H_plain, 'bilinear, no pre-warping', 'r--'),
                        (H_pw, 'bilinear, pre-warped (= cookbook)', 'C0-')]:
    axes[1].plot(freqs / 1000, 20 * np.log10(np.abs(H)), style, label=label)
axes[1].axvline(f0 / 1000, color='0.5', linestyle=':')
axes[1].set_xlabel('Frequency (kHz)')
axes[1].set_ylabel('dB')
axes[1].set_title('Peaking EQ: +6 dB at 10 kHz, Q = 2')
axes[1].legend(loc='upper left')
plt.tight_layout()
plt.show()

for H, label in [(H_analog, 'analog prototype'), (H_plain, 'bilinear, no pre-warping'), (H_pw, 'bilinear, pre-warped')]:
    print(f'{label:<26} peak at {freqs[np.argmax(np.abs(H))]:7.0f} Hz')
print('Cookbook coefficients equal the pre-warped bilinear transform:',
      np.allclose(b_rbj, b_pw / a_pw[0]) and np.allclose(a_rbj, a_pw / a_pw[0]))

Without pre-warping, the peak lands at about 8.7 kHz instead of 10 kHz. With pre-warping it lands exactly on 10 kHz, but the digital curve is still narrower on its high side than the analog one: the frequencies between 10 kHz and infinity are squeezed into 10 to 22.05 kHz. Pre-warping fixes one frequency, not the whole shape. The equalizer in `05_Audio_Effects` uses these cookbook biquads.